<img align="left" src = https://project.lsst.org/sites/default/files/Rubin-O-Logo_0.png width=250 style="padding: 10px"> 
<br>
<b>LSSTCam Quick look</b> <br>
Contact author: Ian Sullivan<br>
Last verified to run: 5 May 2025<br>
LSST Science Piplines version: w_2025_18<br>
Run at USDF on ComCam repo embargo

# 1. Main package imports

In [1]:
import os
import importlib
import pprint

import matplotlib.pyplot as plt
#%matplotlib widget
%matplotlib inline

import math
import numpy as np
import pandas as pd
import scipy
import astropy.units as u
from astropy.coordinates import SkyCoord

In [2]:
import lsst.afw.image as afwImage
import lsst.afw.display as afwDisplay
import lsst.geom
import lsst.afw.geom as afwGeom

import lsst.daf.butler as dafButler
import lsst.pipe.base

In [3]:
from lsst.analysis.ap import apdb
from lsst.analysis.ap import nb_utils
from astropy.table import Table

In [4]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

In [5]:
afwDisplay.setDefaultBackend("firefly")

# Get a butler

In [8]:
repo = '/repo/embargo'

butler_registry = dafButler.Butler(repo)

In [9]:
skymap = "lsst_cells_v1"

In [10]:
skyMap = butler_registry.get("skyMap", collections="skymaps", skymap="lsst_cells_v1")

In [11]:
def find_datasets(butler, dataset, collectionPattern="*"):
    for info in butler_registry.collections.query_info(collectionPattern, include_summary=True):
        if dataset in info.dataset_types:
            print(info.name)

In [12]:
find_datasets(butler_registry, "difference_image", collectionPattern="*LSSTCam*")

LSSTCam/prompt/output-2025-04-27/ApPipe/pipelines-4b333b3-config-8f017ea
LSSTCam/runs/DRP/20250415_20250422/d_2025_04_23/DM-50409/20250427T130850Z
LSSTCam/prompt/output-2025-04-25/ApPipe/pipelines-4b333b3-config-8f017ea
LSSTCam/prompt/output-2025-05-01/ApPipe/pipelines-d71eb52-config-8f017ea
LSSTCam/runs/DRP/20250415_20250422/d_2025_04_23/DM-50409
LSSTCam/prompt/output-2025-04-25
LSSTCam/prompt/output-2025-05-01
LSSTCam/prompt/output-2025-04-27
LSSTCam/runs/DRP/20250415_20250422/d_2025_04_23/DM-50409/hips
u/erykoff/LSSTCam/drpfail/ncf


In [13]:
date = "2025-05-01"
collection_base = "LSSTCam/prompt/output-"+date
butler_registry.registry.queryCollections(collection_base)

['LSSTCam/prompt/output-2025-05-01']

In [14]:
butler = dafButler.Butler(repo, collections=butler_registry.registry.queryCollections(collection_base))

In [15]:
datarefs = butler.query_datasets("difference_image", limit=-100000)

In [16]:
len(datarefs)

40435

In [17]:
visits = []
for ref in datarefs:
    dataId = ref.dataId
    visits.append(dataId['visit'])
visits = set(visits)
visits

{2025050100309,
 2025050100310,
 2025050100311,
 2025050100312,
 2025050100313,
 2025050100314,
 2025050100315,
 2025050100316,
 2025050100317,
 2025050100318,
 2025050100319,
 2025050100320,
 2025050100321,
 2025050100322,
 2025050100323,
 2025050100324,
 2025050100325,
 2025050100326,
 2025050100327,
 2025050100328,
 2025050100329,
 2025050100330,
 2025050100331,
 2025050100332,
 2025050100333,
 2025050100334,
 2025050100335,
 2025050100336,
 2025050100337,
 2025050100338,
 2025050100339,
 2025050100340,
 2025050100341,
 2025050100342,
 2025050100343,
 2025050100344,
 2025050100345,
 2025050100346,
 2025050100347,
 2025050100348,
 2025050100349,
 2025050100350,
 2025050100351,
 2025050100352,
 2025050100353,
 2025050100354,
 2025050100355,
 2025050100356,
 2025050100357,
 2025050100358,
 2025050100359,
 2025050100360,
 2025050100365,
 2025050100366,
 2025050100367,
 2025050100368,
 2025050100369,
 2025050100370,
 2025050100371,
 2025050100372,
 2025050100373,
 2025050100374,
 2025050

In [18]:
def find_templates(science_refs, coadd_refs, skymap=skymap):
    visit_bands = set([(dataref.dataId["visit"], dataref.dataId["band"]) for dataref in science_refs])
    for visit, band in visit_bands:
        refs = [dataref for dataref in science_refs if dataref.dataId["visit"] == visit]
        

In [19]:
def get_xy_from_source_table(table, wcs, degrees=False):
    ra = table['coord_ra']
    dec = table['coord_dec']
    x, y = wcs.skyToPixelArray(ra, dec, degrees=degrees)
    return Table.from_pandas(pd.DataFrame({'x':x, 'y':y}))

In [20]:
def display_diffim(visit, detector, frame=1):
    diffim = butler.get("difference_image", visit=visit, detector=detector)
    # red
    unfiltered = butler.get("dia_source_unfiltered", visit=visit, detector=detector)
    rejected = butler.get("rejected_dia_source", visit=visit, detector=detector)
    trailed = butler.get("long_trailed_source_detector", visit=visit, detector=detector)
    # yellow
    candidate = butler.get("dia_source_unstandardized", visit=visit, detector=detector)
    standardized = butler.get("dia_source_detector", visit=visit, detector=detector)
    
    dia_source = butler.get("dia_source_apdb", visit=visit, detector=detector)
    good = dia_source['reliability'] > 0.1
    # blue
    good_source = dia_source[good]
    # red
    bad_source = dia_source[~good]
    print(f"{len(unfiltered)} unfiltered")
    print(f"{len(trailed)} trailed")
    print(f"{len(candidate)} candidate")
    print(f"{len(bad_source)} low reliability diaSources")
    print(f"{len(good_source)} good diaSources")
    
    # marginal = butler.get("marginal_new_dia_source", visit=visit, detector=detector)
    # ss_source_detector = butler.get("ss_source_detector", visit=visit, detector=detector)
    sky_source = unfiltered["sky_source"]
    
    
    rejected = get_xy_from_source_table(rejected, diffim.wcs)
    candidate = get_xy_from_source_table(candidate, diffim.wcs)
    unfiltered = get_xy_from_source_table(unfiltered[~sky_source], diffim.wcs)
    trailed = get_xy_from_source_table(trailed, diffim.wcs)
    disp = afwDisplay.Display(backend="firefly", frame=frame)
    disp.mtv(diffim)
    for x,y in zip(unfiltered["x"].data,unfiltered["y"].data):
        disp.dot("+", x, y, size=10, ctype="red")
    for x,y in zip(trailed["x"].data,trailed["y"].data):
        disp.dot("x", x, y, size=30, ctype="red")
    for x,y in zip(candidate["x"].data,candidate["y"].data):
        disp.dot("+", x, y, size=10, ctype="yellow")
    for x,y in zip(dia_source["x"].data,dia_source["y"].data):
        disp.dot("+", x, y, size=10, ctype="blue")
    for x,y in zip(good_source["x"].data,good_source["y"].data):
        disp.dot("o", x, y, size=10, ctype="blue")
    for x,y in zip(bad_source["x"].data,bad_source["y"].data):
        disp.dot("o", x, y, size=10, ctype="red")

In [21]:
display_diffim(2025050100492, 100)

162 unfiltered
1 trailed
25 candidate
0 low reliability diaSources
13 good diaSources


In [22]:
def count_diaSources(visit, detector):
    try:
        candidate = butler.get("dia_source_unstandardized", visit=visit, detector=detector)
        dia_source = butler.get("dia_source_apdb", visit=visit, detector=detector)
        print(f"{visit}: {len(candidate)} | {len(dia_source)}")
    except dafButler.DatasetNotFoundError:
        print(f"{visit} detector {detector} not found")

In [23]:
for visit in visits:
    count_diaSources(visit, 100)

2025050100309: 120 | 25
2025050100310: 123 | 17
2025050100311: 211 | 52
2025050100312: 156 | 26
2025050100313: 127 | 11
2025050100314: 118 | 14
2025050100315: 120 | 9
2025050100316: 182 | 29
2025050100317: 74 | 12
2025050100318: 167 | 31
2025050100319: 130 | 24
2025050100320: 141 | 22
2025050100321: 150 | 33
2025050100322: 151 | 38
2025050100323: 139 | 24
2025050100324: 138 | 25
2025050100325: 188 | 49
2025050100326: 109 | 28
2025050100327: 123 | 28
2025050100328: 154 | 24
2025050100329: 131 | 26
2025050100330 detector 100 not found
2025050100331: 136 | 30
2025050100332: 138 | 18
2025050100333: 265 | 55
2025050100334: 168 | 45
2025050100335: 174 | 28
2025050100336 detector 100 not found
2025050100337: 116 | 27
2025050100338: 127 | 23
2025050100339: 101 | 27
2025050100340: 126 | 22
2025050100341 detector 100 not found
2025050100342 detector 100 not found
2025050100343: 156 | 25
2025050100344 detector 100 not found
2025050100345: 169 | 32
2025050100346: 126 | 47
2025050100347: 104 | 39
2

In [24]:
visit = 2025050100321
detector = 100
diffim = butler.get("difference_image", visit=visit, detector=detector)

In [25]:
detector_list = []
for detector in range(190):
    try:
        dia_source = butler.get("dia_source_apdb", visit=visit, detector=detector)
    except Exception as e:
        pass
    else:
        detector_list.append(detector)

In [26]:
def filter_diaSources(visit):
    ng = 0
    nb = 0
    for detector in detector_list:
        try:
            dia_source = butler.get("dia_source_apdb", visit=visit, detector=detector)
            good = dia_source['reliability'] > 0.1
            ng += np.sum(good)
            nb += np.sum(~good)
        except dafButler.DatasetNotFoundError:
            pass
    return(ng, nb)

In [27]:
filter_diaSources(2025050100321)

(np.int64(6428), np.int64(673))

# Full focal plane image

In [28]:
def create_focal_plane_mosaic(butler, exposure_id, image_type="postISRCCD", detectors=None):
    """
    Create a mosaic of all CCDs in the focal plane for a given exposure.
    
    Parameters
    ----------
    butler : Butler
        LSST Butler instance
    exposure_id : int
        Exposure ID to retrieve
    detectors : list, optional
        List of detector IDs to include. If None, uses all detectors.
        
    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure containing the mosaic
    """
    expId = str(exposure_id)
    dayObs = int(expId[:-5])
    seqnum = int(expId[-5:])
    # Get camera geometry if not provided
    camera = LsstCam.getCamera()
    
    if detectors is None:
        detectors = [det.getId() for det in camera]
    
    # Initialize bounds to find the full focal plane extent
    min_x, min_y = float('inf'), float('inf')
    max_x, max_y = float('-inf'), float('-inf')
    
    # First pass: determine focal plane bounds
    for detector_id in detectors:
        detector = camera[detector_id]
        corners = detector.getCorners(FOCAL_PLANE)
        
        for corner in corners:
            min_x = min(min_x, corner.getX())
            min_y = min(min_y, corner.getY())
            max_x = max(max_x, corner.getX())
            max_y = max(max_y, corner.getY())
    
    # Create figure with appropriate aspect ratio
    width = max_x - min_x
    height = max_y - min_y
    aspect_ratio = height / width
    
    fig_width = 15  # inches
    fig = plt.figure(figsize=(fig_width, fig_width * aspect_ratio))
    ax = fig.add_subplot(111)
    
    # Second pass: plot each CCD
    for detector_id in detectors:
        try:
            # Get the postISR CCD image
            ccd = butler.get(image_type, exposure=exposure_id, detector=detector_id)
            
            # Get detector and its position in the focal plane
            detector = camera[detector_id]
            corners = detector.getCorners(FOCAL_PLANE)
            
            # Convert image to normalized array for display
            image_array = ccd.image.array
            image = image_array - np.median(image_array)
            
            # Get detector orientation and position
            orientation = detector.getOrientation()
            center = detector.getCenter(FOCAL_PLANE)
            
            # Create extent for imshow based on detector position
            pixel_size = detector.getPixelSize()
            width_pixels = detector.getBBox().getWidth()
            height_pixels = detector.getBBox().getHeight()
            
            width_mm = width_pixels * pixel_size[0]
            height_mm = height_pixels * pixel_size[1]
            
            # Calculate extent for imshow
            extent = [
                center.getX() - width_mm/2,
                center.getX() + width_mm/2,
                center.getY() - height_mm/2,
                center.getY() + height_mm/2
            ]
            
            # Plot the CCD
            _norm = scl.display.AsinhMapping(minimum=0, stretch=500, Q=10)
            rgb = scl.display.img_to_rgb(image[None], norm=_norm)
            ax.imshow(rgb, extent=extent, origin='lower', cmap='gray', interpolation='antialiased')
            
            # Optionally add detector ID labels
            #ax.text(center.getX(), center.getY(), str(detector_id),
            #       color='red', ha='center', va='center')
            
        except Exception as e:
            print(f"Failed to process detector {detector_id}: {str(e)}")
            continue
    
    # Set axis limits with some padding
    padding = 0.05 * max(width, height)
    ax.set_xlim(min_x - padding, max_x + padding)
    ax.set_ylim(min_y - padding, max_y + padding)
    
    # Add labels and title
    #ax.set_xlabel('Focal Plane X (mm)')
    #ax.set_ylabel('Focal Plane Y (mm)')
    ax.axis('off')
    ax.set_title(f'Focal Plane Mosaic\n{dayObs=}\n{seqnum=}')
    
    return fig